Planteamiento del problema.

El objetivo de este proyecto es analizar los factores que influyen en la duración de los viajes registrados en la Encuesta de Movilidad de Bogotá y municipios cercanos. En particular, se busca estudiar cómo el motivo del viaje, el medio de transporte, las franjas horarias (pico/valle) y la localización origen/destino afectan el tiempo total de desplazamiento, con el fin de explicar y modelar el tiempo de viaje de las personas encuestadas.

Descarga y lectura de datas.

En esta sección se realiza la descarga del dataset desde Kaggle y la lectura del archivo CSV con Pandas. A continuación, se inspecciona la estructura del dataframe (número de filas y columnas, tipos de datos y valores nulos) para identificar las variables relevantes para el problema y posibles necesidades de limpieza.

In [ ]:
import kagglehub

# Descarga de data set
path = kagglehub.dataset_download("eduarmma19/movilidad-de-bogot-caracterizacin-viajes")
print("Path to dataset files:", path)


Using Colab cache for faster access to the 'movilidad-de-bogot-caracterizacin-viajes' dataset.
Path to dataset files: /kaggle/input/movilidad-de-bogot-caracterizacin-viajes


In [ ]:
import os
import pandas as pd

csv_path = os.path.join(path, "Encuesta_movilidad.csv")
print("Ruta completa al CSV:", csv_path)

# Lectura.
df = pd.read_csv(
    csv_path,
    encoding="latin1",
    sep=","
)

# Exploración básica de datos.
print("Shape (filas, columnas):", df.shape)
print(df.head())
print(df.info())


Ruta completa al CSV: /kaggle/input/movilidad-de-bogot-caracterizacin-viajes/Encuesta_movilidad.csv
Shape (filas, columnas): (147251, 33)
   ID_ENCUESTA  NUMERO_PERSONA  NUMERO_VIAJE    MOTIVOVIAJE MUNICIPIO_DESTINO  \
0     18390069               2             1       Tramites   BOGOTA-DC 11001   
1     18390069               2             2  Volver a casa   BOGOTA-DC 11001   
2     18390069               3             1       Estudiar   BOGOTA-DC 11001   
3     18390069               3             2  Volver a casa   BOGOTA-DC 11001   
4     18390891               1             1       Trabajar   BOGOTA-DC 11001   

  DEPARTAMENTO_DESTINO  TIEMPO_CAMINO HORA_INICIO  HORA_FIN  \
0          Bogota D.C.           10.0    08:05:00  09:55:00   
1          Bogota D.C.           10.0    10:21:05  12:11:05   
2          Bogota D.C.           10.0    06:27:00  06:45:00   
3          Bogota D.C.           10.0    08:46:27  09:41:27   
4          Bogota D.C.            7.0    07:47:00  09:19:00 

Selección de variables.

A partir de la exploración inicial(Actividad guiada 1) se observa que el dataframe contiene 33 columnas con información diversa sobre cada viaje. Para centrar el análisis en el tiempo de desplazamiento, se seleccionan aquellas variables más directamente relacionadas con la movilidad (motivo, origen, destino, horarios, modo de transporte y condiciones de operación), dejando fuera, de momento, los factores de expansión y ponderadores que se usarían en análisis de representatividad estadística.


In [ ]:
columnas_relevantes = [
    "ID_ENCUESTA",
    "NUMERO_PERSONA",
    "NUMERO_VIAJE",
    "MOTIVOVIAJE",
    "MUNICIPIO_ORIGEN",
    "DEPARTAMENTO_ORIGEN",
    "MUNICIPIO_DESTINO",
    "DEPARTAMENTO_DESTINO",
    "ZAT_ORIGEN",
    "ZAT_DESTINO",
    "LATITUD_ORIGEN",
    "LONGITUD_ORIGEN",
    "LATITUD_DESTINO",
    "LONGITUD_DESTINO",
    "TIEMPO_CAMINO",
    "HORA_INICIO",
    "HORA_FIN",
    "DIFERENCIA_HORAS",
    "MEDIO_PREDOMINANTE",
    "DIA_HABIL",
    "DIA_NOHABIL",
    "PICO_HABIL",
    "PICO_NOHABIL",
    "VALLE_HABIL",
    "VALLE_NOHABIL"
]

base = df[columnas_relevantes].copy()
base.head()
print("Dimensiones iniciales:", base.shape)

Dimensiones iniciales: (147251, 25)


Manejo de valores nulos.

En esta fase de limpieza del dataset de movilidad de Bogotá, se realizará un diagnóstico de los valores nulos mediante el conteo por variable y su porcentaje respectivo, consolidado en un DataFrame ordenado descendientemente para priorizar intervenciones. Posteriormente, se transfará las columnas de horas de inicio y fin a formato datetime, calculando el tiempo de viaje en minutos con una función personalizada que maneja diferencias negativas cruzando medianoche sumando 24 horas, y se consolidó priorizando TIEMPO_CAMINO cuando existe o el valor calculado de lo contrario. Finalmente, se eliminaran los registros con nulos en TIEMPO_VIAJEMIN y en variables clave como MOTIVOVIAJE, municipios y departamentos de origen/destino, ZATs y coordenadas, reduciendo el dataset de forma controlada y verificando la ausencia total de nulos al cierre del proceso.

In [ ]:
nulos = base.isna().sum()
porcentaje_nulos = (nulos / len(base)) * 100

resumen_nulos = pd.DataFrame({
    "Variable": base.columns,
    "Nulos": nulos.values,
    "Porcentaje (%)": porcentaje_nulos.values
})

print(resumen_nulos.sort_values("Porcentaje (%)", ascending=False))


                Variable   Nulos  Porcentaje (%)
24         VALLE_NOHABIL  145585       98.868599
22          PICO_NOHABIL  144871       98.383712
23           VALLE_HABIL  136312       92.571188
20           DIA_NOHABIL  129521       87.959335
21            PICO_HABIL  125985       85.557993
19             DIA_HABIL   17730       12.040665
14         TIEMPO_CAMINO    4118        2.796585
10        LATITUD_ORIGEN     567        0.385057
11       LONGITUD_ORIGEN     567        0.385057
13      LONGITUD_DESTINO     481        0.326653
12       LATITUD_DESTINO     478        0.324616
8             ZAT_ORIGEN      43        0.029202
9            ZAT_DESTINO      23        0.015620
2           NUMERO_VIAJE       0        0.000000
3            MOTIVOVIAJE       0        0.000000
1         NUMERO_PERSONA       0        0.000000
0            ID_ENCUESTA       0        0.000000
7   DEPARTAMENTO_DESTINO       0        0.000000
6      MUNICIPIO_DESTINO       0        0.000000
5    DEPARTAMENTO_OR

In [ ]:
# Se convierte las horas a datetime
base["HORA_INICIODT"] = pd.to_datetime(base["HORA_INICIO"], format="%H:%M:%S", errors="coerce")
base["HORA_FINDT"] = pd.to_datetime(base["HORA_FIN"], format="%H:%M:%S", errors="coerce")

def calcular_minutos_viaje(h_ini, h_fin):
    if pd.isna(h_ini) or pd.isna(h_fin):
        return pd.NA

    delta = (h_fin - h_ini).total_seconds() / 60

    # Si la diferencia es negativa, se interpreta como viaje que cruza medianoche
    if delta < 0:
        delta = (24 * 60) + delta

    return delta

base["TIEMPO_MINUTOSCALC"] = base.apply(
    lambda row: calcular_minutos_viaje(row["HORA_INICIODT"], row["HORA_FINDT"]),
    axis=1
)

base["TIEMPO_VIAJEMIN"] = base["TIEMPO_CAMINO"].fillna(base["TIEMPO_MINUTOSCALC"])

print("Registros sin TIEMPO_VIAJEMIN:", base["TIEMPO_VIAJEMIN"].isna().sum())
print(base["TIEMPO_VIAJEMIN"].describe())

Registros sin TIEMPO_VIAJEMIN: 1
count     147250.0
unique      1309.0
top            0.0
freq       45720.0
Name: TIEMPO_VIAJEMIN, dtype: float64


In [ ]:
base_antes_nulos = base.shape[0]
base = base.dropna(subset=["TIEMPO_VIAJEMIN"])
base_despues_nulos = base.shape[0]

print("Registros antes de eliminar nulos en TIEMPO_VIAJEMIN:", base_antes_nulos)
print("Registros después:", base_despues_nulos)
print("Registros eliminados:", base_antes_nulos - base_despues_nulos)


Registros antes de eliminar nulos en TIEMPO_VIAJEMIN: 147251
Registros después: 147250
Registros eliminados: 1


In [ ]:
vars_clave = [
    "MOTIVOVIAJE",
    "MUNICIPIO_ORIGEN",
    "DEPARTAMENTO_ORIGEN",
    "MUNICIPIO_DESTINO",
    "DEPARTAMENTO_DESTINO",
    "MEDIO_PREDOMINANTE",
    "ZAT_ORIGEN",
    "ZAT_DESTINO",
    "LATITUD_ORIGEN",
    "LONGITUD_ORIGEN",
    "LATITUD_DESTINO",
    "LONGITUD_DESTINO"
]

print("=== NULOS EN VARIABLES CLAVE (ANTES) ===")
for var in vars_clave:
    if var in base.columns:
        n = base[var].isna().sum()
        pct = (n / len(base)) * 100
        print(f"{var}: {n} nulos ({pct:.2f}%)")

base_antes_vars_clave = base.shape[0]
base = base.dropna(subset=vars_clave)
base_despues_vars_clave = base.shape[0]

print("\nRegistros antes de eliminar nulos en variables clave:", base_antes_vars_clave)
print("Registros después:", base_despues_vars_clave)
print("Registros eliminados:", base_antes_vars_clave - base_despues_vars_clave)

print("\n=== NULOS DESPUÉS DE LIMPIEZA ===")
print(base.isna().sum())

=== NULOS EN VARIABLES CLAVE (ANTES) ===
MOTIVOVIAJE: 0 nulos (0.00%)
MUNICIPIO_ORIGEN: 0 nulos (0.00%)
DEPARTAMENTO_ORIGEN: 0 nulos (0.00%)
MUNICIPIO_DESTINO: 0 nulos (0.00%)
DEPARTAMENTO_DESTINO: 0 nulos (0.00%)
MEDIO_PREDOMINANTE: 0 nulos (0.00%)
ZAT_ORIGEN: 43 nulos (0.03%)
ZAT_DESTINO: 23 nulos (0.02%)
LATITUD_ORIGEN: 567 nulos (0.39%)
LONGITUD_ORIGEN: 567 nulos (0.39%)
LATITUD_DESTINO: 478 nulos (0.32%)
LONGITUD_DESTINO: 481 nulos (0.33%)

Registros antes de eliminar nulos en variables clave: 147250
Registros después: 146324
Registros eliminados: 926

=== NULOS DESPUÉS DE LIMPIEZA ===
ID_ENCUESTA                  0
NUMERO_PERSONA               0
NUMERO_VIAJE                 0
MOTIVOVIAJE                  0
MUNICIPIO_ORIGEN             0
DEPARTAMENTO_ORIGEN          0
MUNICIPIO_DESTINO            0
DEPARTAMENTO_DESTINO         0
ZAT_ORIGEN                   0
ZAT_DESTINO                  0
LATITUD_ORIGEN               0
LONGITUD_ORIGEN              0
LATITUD_DESTINO              0

Transformación de datos.

Fase de estandarización de tipos de datos del dataset de movilidad, se identificarán y conviertirán las columnas numéricas clave como ZATORIGEN, ZATDESTINO, coordenadas geográficas (LATITUD/LONGITUD de origen y destino) y tiempos de viaje (TIEMPOCAMINO, TIEMPOVIAJEMIN) a formato float mediante pd.to_numeric con manejo de errores para coerce a NaN, asegurando precisión decimal y uniformidad. De igualmente, las variables categóricas como MOTIVOVIAJE, nombres de municipios y departamentos, MEDIOPREDOMINANTE y banderas de día hábil/no hábil, pico y valle se transformaron a tipo "categoría" para optimizar memoria y habilitar análisis eficientes en pandas. Finalmente, se verificará la estructura resultante con base.info(), confirmando tipos correctos y eliminando inconsistencias textuales.


In [ ]:
# Numéricos
cols_float = [
    "ZATORIGEN", "ZATDESTINO",
    "LATITUDORIGEN", "LONGITUDORIGEN",
    "LATITUDDESTINO", "LONGITUDDESTINO",
    "TIEMPOCAMINO", "TIEMPOVIAJEMIN"
]

for c in cols_float:
    if c in base.columns:
        base[c] = pd.to_numeric(base[c], errors="coerce")

# Categóricas
cols_cat = [
    "MOTIVOVIAJE",
    "MUNICIPIOORIGEN",
    "DEPARTAMENTOORIGEN",
    "MUNICIPIODESTINO",
    "DEPARTAMENTODESTINO",
    "MEDIOPREDOMINANTE",
    "DIAHABIL",
    "DIANOHABIL",
    "PICOHABIL",
    "PICONOHABIL",
    "VALLEHABIL",
    "VALLENOHABIL"
]

for c in cols_cat:
    if c in base.columns:
        base[c] = base[c].astype("category")

print(base.info())


<class 'pandas.core.frame.DataFrame'>
Index: 146324 entries, 0 to 147250
Data columns (total 29 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   ID_ENCUESTA           146324 non-null  int64         
 1   NUMERO_PERSONA        146324 non-null  int64         
 2   NUMERO_VIAJE          146324 non-null  int64         
 3   MOTIVOVIAJE           146324 non-null  category      
 4   MUNICIPIO_ORIGEN      146324 non-null  object        
 5   DEPARTAMENTO_ORIGEN   146324 non-null  object        
 6   MUNICIPIO_DESTINO     146324 non-null  object        
 7   DEPARTAMENTO_DESTINO  146324 non-null  object        
 8   ZAT_ORIGEN            146324 non-null  float64       
 9   ZAT_DESTINO           146324 non-null  float64       
 10  LATITUD_ORIGEN        146324 non-null  float64       
 11  LONGITUD_ORIGEN       146324 non-null  float64       
 12  LATITUD_DESTINO       146324 non-null  float64       
 13  LONG

Manejo de outliers.

la detección de outliers en el tinempo de viaje se realiza con la regla del rango intercuartico IQR, se calcula Q1, Q2, Q3 e IQR

IQR= Q3-Q1,  y se eliminan los viajes con TIEMPO_VIAJE_MIN menores a
Q1-1.5*IQR o mayores a Q3+1.5*IQR.

In [ ]:
print("Descripción TIEMPO_VIAJEMIN")
print(base["TIEMPO_VIAJEMIN"].describe())

q1 = base["TIEMPO_VIAJEMIN"].quantile(0.25)
q3 = base["TIEMPO_VIAJEMIN"].quantile(0.75)
iqr = q3 - q1

liminf = q1 - 1.5 * iqr
limsup = q3 + 1.5 * iqr

print("Q1:", q1, "Q3:", q3, "IQR:", iqr)
print("Límite inferior:", liminf, "Límite superior:", limsup)

print("Viajes por debajo del límite inferior:",
      (base["TIEMPO_VIAJEMIN"] < liminf).sum())
print("Viajes por encima del límite superior:",
      (base["TIEMPO_VIAJEMIN"] > limsup).sum())

base_limpia = base[
    (base["TIEMPO_VIAJEMIN"] >= liminf) &
    (base["TIEMPO_VIAJEMIN"] <= limsup)
].copy()

print("Shape después de filtrar outliers", base.shape, "-", base_limpia.shape)

Descripción TIEMPO_VIAJEMIN
count     146324.0
unique      1309.0
top            0.0
freq       45423.0
Name: TIEMPO_VIAJEMIN, dtype: float64
Q1: 0.0 Q3: 10.0 IQR: 10.0
Límite inferior: -15.0 Límite superior: 25.0
Viajes por debajo del límite inferior: 0
Viajes por encima del límite superior: 9966
Shape después de filtrar outliers (146324, 29) - (136358, 29)


En la fase final de validación y exportación del dataset de movilidad de Bogotá, se generará un resumen estructurado del dataset limpio (base_limpia) mediante base_limpia.info() para detallar tipos de datos, conteos no nulos y memoria, confirmando la ausencia total de valores faltantes tras las limpiezas previas. Posteriormente, se ejecutaran descriptivas exhaustivas con describe(include="all") que integran estadísticos numéricos (media, desvíos, percentiles) y frecuencias categóricas (top valores, counts), junto con una vista de las primeras filas vía head() para inspección visual rápida.

In [ ]:
print("=== INFO DATASET LIMPIO ===")
print(base_limpia.info())

print("\n=== DESCRIPTIVAS ===")
print(base_limpia.describe(include="all"))

print("\nPrimeras filas:")
print(base_limpia.head())


base_limpia.to_csv("movilidad_bogota_limpia.csv", index=False, encoding="utf-8")
print("\nDataset limpio guardado como 'movilidad_bogota_limpia.csv'")


=== INFO DATASET LIMPIO ===
<class 'pandas.core.frame.DataFrame'>
Index: 136358 entries, 0 to 147250
Data columns (total 29 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   ID_ENCUESTA           136358 non-null  int64         
 1   NUMERO_PERSONA        136358 non-null  int64         
 2   NUMERO_VIAJE          136358 non-null  int64         
 3   MOTIVOVIAJE           136358 non-null  category      
 4   MUNICIPIO_ORIGEN      136358 non-null  object        
 5   DEPARTAMENTO_ORIGEN   136358 non-null  object        
 6   MUNICIPIO_DESTINO     136358 non-null  object        
 7   DEPARTAMENTO_DESTINO  136358 non-null  object        
 8   ZAT_ORIGEN            136358 non-null  float64       
 9   ZAT_DESTINO           136358 non-null  float64       
 10  LATITUD_ORIGEN        136358 non-null  float64       
 11  LONGITUD_ORIGEN       136358 non-null  float64       
 12  LATITUD_DESTINO       136358 non-nu